<a href="https://colab.research.google.com/github/Damainx22/RGV-Business-Survival/blob/main/notebooks/04_bds_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# ============================================
# Notebook 04: BDS Data Cleaning
# Purpose: Load raw BDS data,
#          filter to texas zip codes and FY2018-2022,
#          clean for modeling, save to data/cleaned/
# ============================================

# Mount Google Drive so we can access our raw data files
from google.colab import drive
drive.mount('/content/drive')

# Standard imports
import os
import pandas as pd

# Define folder paths so we don't repeat long strings everywhere
# RAW = where original untouched downloads live
# CLEANED = where filtered/cleaned data gets saved
# MERGED = where the final combined dataset goes (used in notebook 05)
RAW = '/content/drive/MyDrive/rgv_business_survival/data/raw'
CLEANED = '/content/drive/MyDrive/rgv_business_survival/data/cleaned'
MERGED = '/content/drive/MyDrive/rgv_business_survival/data/merged'

# Make sure cleaned folder exists
os.makedirs(CLEANED, exist_ok=True)

# Verify our raw files are accessible
print("Raw files:", sorted(os.listdir(RAW)))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Raw files: ['7a_504_foia_data_dictionary.xlsx', 'bds2023_sec.csv', 'bds2023_st_cty.csv', 'foia-7a-fy2010-fy2019-as-of-251231.csv', 'foia-7a-fy2020-present-as-of-251231.csv', 'zbp18detail.txt', 'zbp18detail.zip', 'zbp19detail.txt', 'zbp19detail.zip', 'zbp20detail.txt', 'zbp20detail.zip', 'zbp21detail.txt', 'zbp21detail.zip', 'zbp22detail.txt', 'zbp22detail.zip']


In [8]:
# Load the Business Dynamics Statistics county-level file
# This file contains firm births, deaths, and job creation/destruction
# for every county in the US from 1978-2023
bds_data = pd.read_csv(f'{RAW}/bds2023_st_cty.csv', low_memory=False)

# Quick check — how many rows and what columns do we have
print(f'Shape: {bds_data.shape} & columns {bds_data.columns}')

Shape: (146924, 27) & columns Index(['year', 'st', 'cty', 'firms', 'estabs', 'emp', 'denom', 'estabs_entry',
       'estabs_entry_rate', 'estabs_exit', 'estabs_exit_rate', 'job_creation',
       'job_creation_births', 'job_creation_continuers',
       'job_creation_rate_births', 'job_creation_rate', 'job_destruction',
       'job_destruction_deaths', 'job_destruction_continuers',
       'job_destruction_rate_deaths', 'job_destruction_rate',
       'net_job_creation', 'net_job_creation_rate', 'reallocation_rate',
       'firmdeath_firms', 'firmdeath_estabs', 'firmdeath_emp'],
      dtype='object')


In [9]:
# Filter to Texas only — state FIPS code for Texas is 48
bds_data = bds_data[bds_data['st'] == 48]

# Filter to our target years 2018-2022 to match other datasets
bds_data = bds_data[bds_data['year'].between(2018, 2022)]

# Keep only the columns relevant to predicting business survival
# estabs_entry_rate = rate of new businesses opening (healthy economy signal)
# estabs_exit_rate = rate of businesses closing (struggling economy signal)
# firmdeath_firms/estabs/emp = firm closures and jobs lost from those closures
bds_data = bds_data[['year', 'st', 'cty', 'firms', 'estabs_entry_rate',
                      'estabs_exit_rate', 'firmdeath_firms', 'firmdeath_estabs',
                      'firmdeath_emp']]
print(bds_data.shape)

(1275, 9)


In [11]:
# Save the cleaned  dataset to data/cleaned/ folder
# index=False prevents pandas from writing row numbers as a column
bds_data.to_csv(f'{CLEANED}/bds_texas_clean.csv', index=False)

# Confirm save was successful by printing shape and first few rows
print(f'Saved: {len(bds_data)} rows, {len(bds_data.columns)} columns')
print(bds_data.head())

Saved: 1275 rows, 9 columns
        year  st  cty firms estabs_entry_rate estabs_exit_rate  \
130327  2018  48    1   753             7.724            9.031   
130328  2018  48    3   366            14.209            7.775   
130329  2018  48    5  1553             6.559            7.973   
130330  2018  48    7   430             8.237           19.007   
130331  2018  48    9   180            10.106           11.170   

       firmdeath_firms firmdeath_estabs firmdeath_emp  
130327              53               55           314  
130328              19               19            95  
130329              80               80           429  
130330              57               57           287  
130331              15               15            57  
